### Import Libraries

In [ ]:
# You already have these two:
# 1. SFT model  — mistralai/Mistral-7B-Instruct-v0.2  (or your own trained SFT)
# 2. Reward model — e.g. OpenAssistant/reward-model-deberta-v3-large-v2
#                   or any HuggingFace model with a classification head

# pip install trl transformers peft bitsandbytes datasets accelerate

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, TaskType, get_peft_model
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead
from datasets import load_dataset

SFT_MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"
RM_MODEL_NAME  = "OpenAssistant/reward-model-deberta-v3-large-v2"
PPO_OUTPUT     = "./checkpoints/ppo-output"

### Load the reward model (frozen — never trained)

In [ ]:
# ── Reward model ──────────────────────────────────────────────
# Loaded once, set to eval(), never gets gradients
# This model takes (prompt + response) text and returns a scalar score

rm_tokenizer = AutoTokenizer.from_pretrained(RM_MODEL_NAME)

reward_model = AutoModelForSequenceClassification.from_pretrained(
    RM_MODEL_NAME,
    torch_dtype=torch.bfloat16,
).eval().to("cuda")   # separate device or same — just never call .train() on it

# Quick sanity check before PPO starts
def get_reward(prompt: str, response: str) -> float:
    """Returns scalar reward score for a (prompt, response) pair."""
    # Most HuggingFace reward models expect "Human: ...\nAssistant: ..." format
    text = f"Human: {prompt}\nAssistant: {response}"
    inputs = rm_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to(reward_model.device)
    with torch.no_grad():
        score = reward_model(**inputs).logits[0].item()
    return score

# Test it — chosen should score higher than rejected
print(get_reward("What is 2+2?", "The answer is 4."))           # e.g. 2.8
print(get_reward("What is 2+2?", "I have no idea whatsoever"))  # e.g. -1.2

### Load the SFT model as policy + reference

In [ ]:
# ── Policy model ──────────────────────────────────────────────
# AutoModelForCausalLMWithValueHead wraps the SFT model and adds
# a small linear "value head" on top for PPO's advantage estimation.
# The value head is the only NEW parameters — everything else is SFT weights.

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

policy = AutoModelForCausalLMWithValueHead.from_pretrained(
    SFT_MODEL_NAME,
    quantization_config=bnb,
    device_map="auto",
    peft_config=LoraConfig(       # optional LoRA — keeps VRAM manageable
        r=16, lora_alpha=32,
        target_modules=["q_proj","k_proj","v_proj","o_proj"],
        lora_dropout=0.05, bias="none",
        task_type=TaskType.CAUSAL_LM,
    ),
)

# ── Reference model ────────────────────────────────────────────
# Exact same weights as policy starting point — frozen forever.
# TRL's PPOTrainer uses this to compute KL[π_θ ‖ π_ref] each step.

ref_model = AutoModelForCausalLMWithValueHead.from_pretrained(
    SFT_MODEL_NAME,
    quantization_config=bnb,
    device_map="auto",
)
# Make absolutely sure ref never gets gradients
for param in ref_model.parameters():
    param.requires_grad = False

# ── Tokenizer ──────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(SFT_MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"   # critical for PPO — generation needs left padding

### Configure PPO

In [ ]:
# Every hyperparameter explained inline

ppo_config = PPOConfig(
    output_dir=PPO_OUTPUT,

    # ── Batch sizes ──
    batch_size=16,          # prompts collected per PPO iteration (rollout batch)
    mini_batch_size=4,      # gradient update chunk size — batch_size must be divisible by this
    gradient_accumulation_steps=1,

    # ── Learning ──
    learning_rate=1e-5,     # lower than SFT — RL is noisy, large LR diverges
    ppo_epochs=4,           # how many gradient passes over the same rollout batch
                            # more passes = more efficient but rollouts go stale

    # ── KL penalty ──
    kl_penalty="kl",        # "kl" = full KL divergence, "abs" = |KL|, "mse" = KL²
    init_kl_coef=0.2,       # β — starting weight of KL penalty term
    adap_kl_ctrl=True,      # auto-adjust β during training to hit target_kl
    target_kl=6.0,          # desired KL between policy and ref each step
                            # if KL > target → increase β (tighten leash)
                            # if KL < target → decrease β (loosen leash)

    # ── PPO clipping ──
    cliprange=0.2,          # ε in clip(ratio, 1-ε, 1+ε) — almost always 0.2
    cliprange_value=0.2,    # same clip for value function loss

    # ── Value function ──
    vf_coef=0.1,            # weight of value function loss in total loss

    # ── Stability ──
    max_grad_norm=1.0,      # gradient clipping — prevents exploding gradients

    # ── Logging ──
    log_with=None,          # set to "wandb" if you want experiment tracking
    logging_steps=10,
)

ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=policy,           # trainable policy with value head
    ref_model=ref_model,    # frozen reference — TRL handles KL computation
    tokenizer=tokenizer,
)

### Dataset — prompts only

In [ ]:
# PPO only needs prompts — no labels, no chosen/rejected
# The policy generates the responses live during training

raw_dataset = load_dataset(
    "trl-internal-testing/hh-rlhf-helpful-base-trl-style",
    split="train"
)
# dataset = load_dataset("HuggingFaceH4/ultrachat_200k", split="train_sft")

# Extract just the prompt column
prompts = [item["prompt"] for item in raw_dataset.select(range(4000))]

# Apply chat template so prompts match what the Instruct model expects
def format_prompt(prompt_text: str) -> str:
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt_text}],
        tokenize=False,
        add_generation_prompt=True,  # adds the [/INST] opening for assistant turn
    )

formatted_prompts = [format_prompt(p) for p in prompts]

### The PPO training loop — every step annotated

In [ ]:
from torch.utils.data import DataLoader

gen_kwargs = {
    "max_new_tokens": 200,
    "do_sample": True,
    "temperature": 0.7,
    "top_p": 0.9,
    "pad_token_id": tokenizer.eos_token_id,
}

num_epochs = 1

for epoch in range(num_epochs):
    for batch_start in range(0, len(formatted_prompts), ppo_config.batch_size):

        # ── STEP 1: Sample batch of prompts ─────────────────────────────
        batch_prompts = formatted_prompts[
            batch_start : batch_start + ppo_config.batch_size
        ]
        if len(batch_prompts) < ppo_config.batch_size:
            break  # skip incomplete final batch

        # ── Tokenize prompts into tensors ────────────────────────────────
        # PPOTrainer.generate() expects a list of 1D tensors (not batched)
        query_tensors = []
        for p in batch_prompts:
            ids = tokenizer.encode(p, return_tensors="pt").squeeze(0)
            query_tensors.append(ids.to(policy.pretrained_model.device))

        # ── STEP 2: Policy generates responses (rollout) ─────────────────
        # policy generates response tokens autoregressively
        # internally records log π_θ(token) for each generated token
        # response_tensors contains ONLY the new tokens (not the prompt)
        response_tensors = ppo_trainer.generate(
            query_tensors,
            return_prompt=False,   # response_tensors = generated tokens only
            **gen_kwargs,
        )

        # Decode responses to text for reward model
        responses_text = [
            tokenizer.decode(r, skip_special_tokens=True)
            for r in response_tensors
        ]

        # ── STEP 3: Reward model scores each response ────────────────────
        # For each (prompt, response) pair, get a scalar reward score
        # The reward model never receives gradients — purely an oracle
        rewards = []
        for prompt_text, response_text in zip(prompts[batch_start:batch_start+ppo_config.batch_size], responses_text):
            score = get_reward(prompt_text, response_text)  # our function from above
            rewards.append(torch.tensor(score, dtype=torch.float32))

        # rewards is a list of scalar tensors, e.g. [tensor(2.1), tensor(-0.4), ...]

        # ── STEPS 4-7: PPO update (TRL handles internally) ───────────────
        # Inside ppo_trainer.step(), TRL does ALL of the following:
        #
        # STEP 4 — KL penalty:
        #   ref_model computes log π_ref(response | prompt) for each pair
        #   KL = mean(log π_θ - log π_ref) per token, summed over response
        #
        # STEP 5 — Value head:
        #   policy's value head V(s) estimates expected reward at each token
        #
        # STEP 6 — Combined reward + advantage:
        #   R_t = reward − β · KL_t   (only at final token, rest are KL-penalized)
        #   A_t = R_t − V_t (advantage = actual − expected)
        #   GAE (Generalized Advantage Estimation) smooths across timesteps
        #
        # STEP 7 — PPO clipped gradient update (ppo_epochs passes):
        #   ratio = π_θ_new(a|s) / π_θ_old(a|s)   (new vs old policy probs)
        #   L_CLIP = min(ratio·A, clip(ratio, 1-ε, 1+ε)·A)
        #   L_VF   = (V_predicted − V_target)²      (value function loss)
        #   L_ENT  = -entropy bonus                  (encourages exploration)
        #   L_total = -L_CLIP + vf_coef·L_VF + ent_coef·L_ENT
        #   Backprop + optimizer step on policy + value head

        stats = ppo_trainer.step(
            query_tensors,      # list of prompt token tensors
            response_tensors,   # list of response token tensors
            rewards,            # list of scalar reward tensors
        )

        # ── STEP 8: Log and repeat ────────────────────────────────────────
        mean_reward = torch.stack(rewards).mean().item()
        kl          = stats.get("objective/kl", 0)
        policy_loss = stats.get("ppo/loss/policy", 0)
        value_loss  = stats.get("ppo/loss/value", 0)

        step_num = batch_start // ppo_config.batch_size
        print(
            f"Epoch {epoch} | Step {step_num:4d} | "
            f"Reward: {mean_reward:+.3f} | "
            f"KL: {kl:.3f} | "
            f"Policy loss: {policy_loss:.4f} | "
            f"Value loss: {value_loss:.4f}"
        )

        # Watch KL — if it keeps growing past target_kl, β is being increased
        # If reward keeps going up but KL is stable, training is healthy
        if kl > 20:
            print("WARNING: KL divergence too high — policy drifting from SFT!")

# ── Save final model ──────────────────────────────────────────────────────────
ppo_trainer.save_pretrained(PPO_OUTPUT)
tokenizer.save_pretrained(PPO_OUTPUT)
print(f"PPO model saved → {PPO_OUTPUT}")

### Load and predict from the saved PPO model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import AutoModelForCausalLMWithValueHead
import torch

# Option A — load with value head (exact checkpoint)
model_with_head = AutoModelForCausalLMWithValueHead.from_pretrained(
    "./checkpoints/ppo-output",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
# For inference, access the underlying LM directly
lm = model_with_head.pretrained_model

# Option B — load just the LM (cleaner, value head not needed at inference)
lm = AutoModelForCausalLM.from_pretrained(
    "./checkpoints/ppo-output",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained("./checkpoints/ppo-output")

def predict(prompt: str, max_new_tokens: int = 256) -> str:
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(lm.device)
    with torch.no_grad():
        out = lm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )

print(predict("What is the best way to learn deep learning?"))